# 1IHM - Norwalk virus (norovirus) capsid

The deposited entry is the T=3 icosahedral capsid: 180 copies of the VP1 coat protein,
arranged as 60 copies of the A/B/C quasi-equivalent trimer. The target assembly is
therefore a 180-mer, and the interesting question is whether a NERDSS model built
straight from the structure nucleates and grows toward it.

Run without ProAffinity: `predict_affinity` is left at its default `False`, so every
binding reaction gets the generic `default_on_rate_3d_ka = 120.0` and the off rate
implied by it, rather than a per-interface predicted affinity.

In [ ]:
# Path handling (standard library)
from pathlib import Path

# Core imports
import ionerdss as ion
from ionerdss import build_system_from_pdb

# For visualizations
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
pdb_id = "1ihm"

# Build the system using simplified API
# This takes ~45 s: the biological assembly is a 77 MB mmCIF with 180 chains.
system = build_system_from_pdb(
    source=pdb_id,
    workspace_path=f"{pdb_id}_dir",

    # The asymmetric unit is only the A/B/C protomer and yields 3 binding
    # reactions -- not enough contacts to close a shell. The biological
    # assembly carries all 180 chains and recovers the full contact set.
    pdb_file_format="bioassembly1",

    # Interface detection: 0.6 nm / 3 residues is the ioNERDSS default and is
    # already the right choice here. Dropping to 0.4 nm finds a single contact;
    # 0.8 nm over-splits into 17 interfaces / 11 reactions. Between 0.6 and 0.8
    # the answer is stable at 13 interfaces / 7 reactions whether the residue
    # cutoff is 2 or 3, which is the sign of a well-conditioned choice.
    interface_detect_distance_cutoff=0.6,
    interface_detect_n_residue_cutoff=3,

    # All 180 chains are the same VP1 sequence, so they collapse to one
    # molecule type carrying all 13 interfaces.
    chain_grouping_seq_threshold=0.5,

    # Capsid: project the subunits onto a best-fit sphere so the association
    # geometry is consistent with a closed shell.
    is_on_sphere=True,

    # 360 subunits = two capsids' worth, ~4.8 uM in a 500 nm box.
    nerdss_water_box=[500.0, 500.0, 500.0],
    nerdss_total_molecule_count=360,
    nerdss_n_itr=300000,

    # Transition matrix. NERDSS indexes this matrix by complex size with no
    # bounds check, so it MUST be >= the total molecule count or the run
    # segfaults as soon as a complex outgrows it.
    count_transition=True,
    transition_matrix_size=400,
)

In [ ]:
# List all generated files
workspace_path = Path(f"{pdb_id}_dir")

print("Generated Files:")

print("\n NERDSS Input Files:")
nerdss_dir = workspace_path / "nerdss_files"
if nerdss_dir.exists():
    for file in sorted(nerdss_dir.glob("*.mol")) + sorted(nerdss_dir.glob("*.inp")):
        size = file.stat().st_size / 1024  # KB
        print(f"  nerdss_files/{file.name:<30} ({size:>6.1f} KB)")

print("\n System Data:")
outputs_dir = workspace_path / "outputs" / "systems"
if outputs_dir.exists():
    for file in sorted(outputs_dir.glob("*.json")):
        size = file.stat().st_size / 1024  # KB
        print(f"  outputs/systems/{file.name:<27} ({size:>6.1f} KB)")

print("\n System Builder Log:")
logs_dir = workspace_path / "logs"
if logs_dir.exists():
    for file in sorted(logs_dir.glob("*.log")):
        size = file.stat().st_size / 1024  # KB
        print(f"  logs/{file.name:<38} ({size:>6.1f} KB)")

# The binding reactions ioNERDSS derived from the structure
print("\n Reaction network:")
print((nerdss_dir / "parms.inp").read_text().split("start reactions")[1])

In [ ]:
# run NERDSS with subprocess
import subprocess

# Check if NERDSS is available
# nerdss_cmd should be replaced with the actual path to the NERDSS executable
nerdss_cmd = "PATH_TO_NERDSS_REPO/bin/nerdss"
nerdss_path = Path(nerdss_cmd).expanduser() # replaces tilde with appropriate user home path

if nerdss_path.exists():

    # Run NERDSS
    result = subprocess.run(
        f"{nerdss_cmd} -f parms.inp",
        shell=True,
        cwd=f"{pdb_id}_dir/nerdss_files",
        capture_output=True,
        text=True
    )

    if result.returncode == 0:
        print("NERDSS simulation completed!")
        print(f"\nCheck {pdb_id}_dir/nerdss_files/ for output files")
    else:
        # returncode 139 here almost always means a complex grew past
        # transition_matrix_size -- NERDSS indexes its transition matrix by
        # complex size without a bounds check, so keep that value >= the
        # total molecule count.
        print(f"NERDSS simulation failed (returncode {result.returncode})")
        print(result.stderr[:500])
else:
    print("NERDSS not found at:", nerdss_cmd)

In [ ]:
# Initialize Analyzer with NERDSS output directory
analysis = ion.Analyzer(f"{pdb_id}_dir")

print(f"Found {len(analysis.simulations)} simulation(s)")
for i, sim in enumerate(analysis.simulations):
    print(f"  [{i}] Simulation ID: {sim.id}")

sim = analysis.get_simulation(0)

# Free subunits and the small oligomers they pass through on the way up
complex_compositions = [{"A": n} for n in [1, 2, 3, 5, 10, 30, 60, 90, 180]]
time, counts = sim.get_time_series(complex_compositions)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for i, comp in enumerate(complex_compositions):
    axes[0].plot(time, counts[i], label=f"A{comp['A']}")
axes[0].set_xlabel("time (s)")
axes[0].set_ylabel("copy number")
axes[0].set_title("1IHM: complex counts")
axes[0].legend(ncol=2, fontsize=8)

# Growth of the assembly, against the 180-mer target
t_max, largest = sim.get_largest_size_time_series()
t_avg, average = sim.get_average_size_time_series()
axes[1].plot(t_max, largest, label="largest complex")
axes[1].plot(t_avg, average, label="average complex")
axes[1].axhline(180, ls="--", c="k", lw=1, label="T=3 capsid (180)")
axes[1].set_xlabel("time (s)")
axes[1].set_ylabel("subunits per complex")
axes[1].set_title("1IHM: assembly growth")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"largest complex reached: {np.nanmax(largest):.0f} subunits "
      f"({np.nanmax(largest) / 180:.0%} of a T=3 capsid)")
print(f"free subunits remaining: {counts[0][-1]:.0f} of 360")